# PC06 — Chiến lược, KPI và bàn giao kết quả cuối

> **Cách đọc:** Kết quả và nhận xét chính hiển thị trực tiếp. Nhấn **Xem chi tiết** để mở bảng đối chiếu; nhật ký tiến trình cũng được thu gọn. Các phép tính và dữ liệu vẫn được giữ đầy đủ.

**Mục tiêu:** chuyển hồ sơ mô tả thành giả thuyết hành động có thể đo lường. Chỉ dùng mô hình huấn luyện trong phiên; không huấn luyện hoặc đổi nhãn tại bước chiến lược.

Đọc sau PC01–PC05. Giá trị cao là Net delivered tích lũy, không phải lợi nhuận, tiền thực nhận hay CLV. Mốc 31/12/2022 là mốc nghiên cứu, không dùng trực tiếp để gọi khách đang rời bỏ tại hiện tại.

In [1]:
from pathlib import Path
import sys, os, json
sys.dont_write_bytecode = True
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'notebook/ham_dung_chung.py').is_file())
if str(ROOT/'notebook') not in sys.path:sys.path.insert(0,str(ROOT/'notebook'))
sys.dont_write_bytecode = True
# Nạp định nghĩa hỗ trợ riêng, không sửa file Python dùng chung của nhóm.
_support_path = ROOT / 'notebook/05_phan_khuc_khach_hang/00_ham_ho_tro_phan_cum.ipynb'
_support_doc = json.loads(_support_path.read_text(encoding='utf-8'))
for _cell_index, _cell in enumerate(_support_doc['cells']):
    if _cell['cell_type'] == 'code':
        exec(compile(''.join(_cell['source']), f'{_support_path}:cell{_cell_index+1}', 'exec'), globals())

Study = phan_cum_tich_hop().Study
MODEL_LABEL = os.environ.get('DATATHON_MODEL_RUN','combined_v1')
study = Study(ROOT,'PC06')
manifest = study.use_model(MODEL_LABEL)
a = study.a
profiles = pd.read_csv(study.modelbase/'cluster_profiles.csv')
study.table('final_profiles','Hồ sơ mô hình được chọn thực tế',profiles);


,Nhóm,Số khách,R trung vị,F trung vị,M trung vị,Tổng Net,TB tỷ trọng cancelled/khách,TB tỷ trọng returned/khách,TB tỷ trọng có returns/khách,TB tỷ trọng thiếu shipment/khách,Số SP khác nhau trung vị,Tỷ trọng khách,Tỷ trọng Net
0,0,34498,2307.0,1.0,29651.965,1.435208e+09,0.061616,0.036944,0.036891,0.000000,2.0,0.405310,0.114650
1,1,40342,598.0,5.0,124473.995,5.614874e+09,0.075082,0.045350,0.045252,0.001875,6.0,0.473971,0.448538
2,2,10275,244.0,19.0,477244.080,5.468093e+09,0.082397,0.050480,0.050340,0.000975,20.0,0.120719,0.436812


## PC06.1. Tên nhóm dựa trên kết quả, không cố định số nhãn

**Câu hỏi: đặt tên nhóm thế nào để không phụ thuộc mã cụm?**

Mã 0, 1, 2 chỉ giúp lưu và ghép kết quả. Đọc đồng thời Recency, Frequency và Monetary trung vị để đặt tên mô tả. Giá trị cao ở đây là tiền mua tích lũy cao trong tập nghiên cứu, chưa là lợi nhuận hoặc tiềm năng tương lai.

Nếu một lượt mới đổi số cụm hoặc thứ tự đặc điểm không còn giống trước, phải xem lại tên. Không ép ba tên cũ vào một kết quả mới chỉ vì mã nhóm nhìn tương tự.


In [2]:
ranked = profiles.sort_values('Monetary_median')
if len(ranked)!=3:raise ValueError('Chiến lược ba mức cần thẩm định lại khi k thay đổi')
assert ranked.Frequency_median.is_monotonic_increasing
assert ranked.Recency_median.is_monotonic_decreasing
names=['Lâu chưa mua, giá trị tích lũy thấp','Mức hoạt động và giá trị trung gian','Mua nhiều, giá trị tích lũy cao']
segment_map=dict(zip(ranked.cluster,names))
final=a[['customer_id','Recency','Frequency','Monetary','cluster']].copy()
final['segment']=final.cluster.map(segment_map)
assert final.customer_id.is_unique and final.segment.notna().all()
# Chỉ lưu kết quả phân cụm chính, không xuất bảng/ảnh thử nghiệm.
phan_cum_tich_hop().store_final(ROOT,MODEL_LABEL,final)
study.table('segment_sizes','Quy mô và tỷ trọng Net theo tên nhóm',profiles.assign(segment=profiles.cluster.map(segment_map)));


,Nhóm,Số khách,R trung vị,F trung vị,M trung vị,Tổng Net,TB tỷ trọng cancelled/khách,TB tỷ trọng returned/khách,TB tỷ trọng có returns/khách,TB tỷ trọng thiếu shipment/khách,Số SP khác nhau trung vị,Tỷ trọng khách,Tỷ trọng Net,segment
0,0,34498,2307.0,1.0,29651.965,1.435208e+09,0.061616,0.036944,0.036891,0.000000,2.0,0.405310,0.114650,"Lâu chưa mua, giá trị tích lũy thấp"
1,1,40342,598.0,5.0,124473.995,5.614874e+09,0.075082,0.045350,0.045252,0.001875,6.0,0.473971,0.448538,Mức hoạt động và giá trị trung gian
2,2,10275,244.0,19.0,477244.080,5.468093e+09,0.082397,0.050480,0.050340,0.000975,20.0,0.120719,0.436812,"Mua nhiều, giá trị tích lũy cao"


## PC06.2. Đề xuất chương trình theo nhóm

**Câu hỏi: có thể thử những hành động nào từ hồ sơ nhóm?**

Nhóm mua nhiều có thể được thử chương trình duy trì trải nghiệm; nhóm trung gian có thể được thử gợi ý mua tiếp; nhóm lâu chưa mua có thể được thử tái kích hoạt có chọn lọc. Mỗi đề xuất cần mục tiêu, chi phí và chỉ số đo riêng.

RFM không chứa sở thích, nên cá nhân hóa sản phẩm phải kiểm thêm lịch sử danh mục. Không suy khách thích giảm giá hoặc ưu đãi chắc chắn tăng mua. Trước tiếp cận cần xác nhận dữ liệu còn hiệu lực và điều kiện liên hệ.


In [3]:
strategies=pd.DataFrame([
{'segment':names[0],'Mục tiêu':'Tái kích hoạt có chọn lọc','Hành động thử':'Thông điệp nhắc mua với ngân sách giới hạn; khảo sát nhu cầu','KPI chính':'Chênh lệch tỷ lệ mua trong 60 ngày so với đối chứng','Rủi ro':'Lịch sử cũ, sai thời điểm; ưu đãi có thể tốn chi phí'},
{'segment':names[1],'Mục tiêu':'Khuyến khích mua tiếp phù hợp chu kỳ','Hành động thử':'Gợi ý bổ sung từ danh mục đã mua; thử thông điệp/ưu đãi riêng','KPI chính':'Net/khách đủ điều kiện trong 60 ngày và chi phí tăng thêm','Rủi ro':'Không biết sở thích từ RFM; phân nhóm không chứng minh nhạy giá'},
{'segment':names[2],'Mục tiêu':'Duy trì trải nghiệm và mua lại','Hành động thử':'Chăm sóc ưu tiên, hỗ trợ và quyền lợi có kiểm soát','KPI chính':'Chênh lệch Net/khách và tỷ lệ mua trong 60 ngày','Rủi ro':'M cao không bảo đảm lợi nhuận; ưu đãi có thể giảm biên'}])
counts=final.segment.value_counts().rename_axis('segment').reset_index(name='customers')
study.table('strategies','Giả thuyết hành động, chưa phải hiệu quả đã đạt',strategies.merge(counts,on='segment',validate='one_to_one'));


,segment,Mục tiêu,Hành động thử,KPI chính,Rủi ro,Số khách
0,"Lâu chưa mua, giá trị tích lũy thấp",Tái kích hoạt có chọn lọc,Thông điệp nhắc mua với ngân sách giới hạn; kh...,Chênh lệch tỷ lệ mua trong 60 ngày so với đối ...,"Lịch sử cũ, sai thời điểm; ưu đãi có thể tốn c...",34498
1,Mức hoạt động và giá trị trung gian,Khuyến khích mua tiếp phù hợp chu kỳ,Gợi ý bổ sung từ danh mục đã mua; thử thông đi...,Net/khách đủ điều kiện trong 60 ngày và chi ph...,Không biết sở thích từ RFM; phân nhóm không ch...,40342
2,"Mua nhiều, giá trị tích lũy cao",Duy trì trải nghiệm và mua lại,"Chăm sóc ưu tiên, hỗ trợ và quyền lợi có kiểm ...",Chênh lệch Net/khách và tỷ lệ mua trong 60 ngày,M cao không bảo đảm lợi nhuận; ưu đãi có thể g...,10275


## PC06.3. Định nghĩa KPI và thiết kế thử nghiệm

**Câu hỏi: đo hiệu quả chiến lược thế nào khi có dữ liệu triển khai?**

Đây là thiết kế đề xuất: chốt khách đủ điều kiện trước khi bắt đầu, chia ngẫu nhiên trong từng phân khúc vào nhóm thử nghiệm và đối chứng, theo dõi cùng 60 ngày. Đánh giá theo nhóm được phân ban đầu, kể cả khách không mở tin hoặc không mua.

Tỷ lệ mua lấy số khách có đơn delivered trong kỳ chia toàn bộ khách được phân nhóm. Số đơn/khách và Net/khách dùng cùng mẫu số. Tác động là chênh lệch giữa nhóm thử nghiệm và đối chứng trong cùng kỳ, không chỉ tỷ lệ của nhóm nhận ưu đãi.

Cần xác định trước cỡ mẫu, ngân sách, thời gian, chi phí và mức thay đổi đáng quan tâm. Báo khoảng tin cậy và hủy/trả; nếu không có người quay lại, chi phí trên khách quay lại không xác định, không ghi bằng 0. Chưa có dữ liệu sau triển khai để điền kết quả thực tế.


In [4]:
kpis=pd.DataFrame([
['Tỷ lệ mua trong kỳ','Khách có đơn delivered trong 60 ngày / toàn bộ khách được phân nhóm ban đầu'],
['Số đơn/khách','Đơn delivered trong 60 ngày / toàn bộ khách được phân nhóm ban đầu'],
['Net/khách','Net delivered trong 60 ngày / toàn bộ khách được phân nhóm ban đầu'],
['Tác động tỷ lệ mua','Tỷ lệ nhóm thử - tỷ lệ nhóm đối chứng, cùng kỳ và định nghĩa'],
['Chi phí/khách quay lại','Chi phí / khách có đơn lại; NA nếu không có khách; chưa là chi phí tăng thêm']
],columns=['KPI','Định nghĩa'])
study.table('kpi_definitions','Định nghĩa để triển khai khi có dữ liệu mới',kpis);


,KPI,Định nghĩa
0,Tỷ lệ mua trong kỳ,Khách có đơn delivered trong 60 ngày / toàn bộ...
1,Số đơn/khách,Đơn delivered trong 60 ngày / toàn bộ khách đư...
2,Net/khách,Net delivered trong 60 ngày / toàn bộ khách đư...
3,Tác động tỷ lệ mua,"Tỷ lệ nhóm thử - tỷ lệ nhóm đối chứng, cùng kỳ..."
4,Chi phí/khách quay lại,Chi phí / khách có đơn lại; NA nếu không có kh...


## PC06.4. Bàn giao và mức sẵn sàng

**Câu hỏi: nghiên cứu hiện sẵn sàng dùng đến mức nào?**

Đã có phân cụm mô tả, đối chứng thuật toán, hồ sơ, kiểm độ nhạy và kế hoạch KPI. Chưa có đánh giá theo thời gian tương lai hoặc bằng chứng chương trình chăm sóc tạo tác động.

Chạy PC01 → PC06; các notebook tự nạp hỗ trợ 00. Notebook hỗ trợ 00 là mã nguồn cần giữ; Không có nhánh xuất báo cáo phụ. Những notebook kinh doanh/dự báo của nhóm được giữ nguyên, không có nghĩa tất cả đã được tái nghiệm thu trong lần sửa này.


In [5]:
study.finish()

{'part': 'PC06',
 'state': 'completed',
 'model_run': 'combined_v1',
 'tables': 4,
 'figures': 0}